In [ ]:
!pip install -q scikit-learn==1.9.0 xgboost==3.2.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 126.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.7/131.7 MB 19.9 MB/s eta 0:00:00


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Router Export Script (run on Colab, NOT on the Jetson).

Trains every router artifact the Jetson script needs -- bench routers (for
overhead measurement), 2-tier routers (8 seeds x 2 datasets x 5 families),
and 3-tier routers (A->C, A->B, B->C sub-routers x 8 seeds x 2 datasets x 5
families) -- and saves them to disk via joblib. Copy the resulting
`--router_dir` folder to the Jetson (e.g. via scp or a USB drive) and point
idk_cascade_jetson_nano_edition_LOAD_ONLY.py at it with --router_dir.

No evaluation, latency measurement, or accuracy computation happens here --
that all still needs to happen ON the Jetson itself, since the whole point of
running there is to get real embedded-hardware numbers. This script only
produces the trained router *weights*.
"""

import argparse
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
import torch
import torchvision
import torchvision.transforms as transforms
import xgboost as xgb

EARLY_EXIT_THRESHOLD = 0.90
DEFAULT_SEEDS_2TIER = [1, 2, 3, 4, 5, 6, 7, 8]
DEFAULT_SEEDS_3TIER = [42, 43, 44, 45, 46, 47, 48, 49]

DATASETS = {
    'CIFAR-100': {
        'transform': transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5071, 0.4867, 0.4408], std=[0.2675, 0.2565, 0.2761])
        ]),
    },
    'CIFAR-10': {
        'transform': transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.4914, 0.4822, 0.4465], std=[0.2471, 0.2435, 0.2616])
        ]),
    }
}

def get_dataset(dataset_name, transform, data_root='./data'):
    if dataset_name == 'CIFAR-100':
        return torchvision.datasets.CIFAR100(root=data_root, train=False, download=True, transform=transform)
    elif dataset_name == 'CIFAR-10':
        return torchvision.datasets.CIFAR10(root=data_root, train=False, download=True, transform=transform)
    raise ValueError("Unknown dataset: {}".format(dataset_name))

def load_models_for_dataset(dataset_name, device):
    if dataset_name == 'CIFAR-100':
        a = torch.hub.load("chenyaofo/pytorch-cifar-models", "cifar100_resnet20", pretrained=True).to(device)
        b = torch.hub.load("chenyaofo/pytorch-cifar-models", "cifar100_resnet32", pretrained=True).to(device)
        c = torch.hub.load("chenyaofo/pytorch-cifar-models", "cifar100_resnet56", pretrained=True).to(device)
    elif dataset_name == 'CIFAR-10':
        a = torch.hub.load("chenyaofo/pytorch-cifar-models", "cifar10_resnet20", pretrained=True).to(device)
        b = torch.hub.load("chenyaofo/pytorch-cifar-models", "cifar10_resnet32", pretrained=True).to(device)
        c = torch.hub.load("chenyaofo/pytorch-cifar-models", "cifar10_resnet56", pretrained=True).to(device)
    else:
        raise ValueError("Unknown dataset: {}".format(dataset_name))
    a.eval(); b.eval(); c.eval()
    return a, b, c

def extract_telemetry(loader, model_a, model_c, device):
    rows = []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            probs_a = torch.softmax(model_a(images), dim=1)
            conf_A, preds_a = torch.max(probs_a, dim=1)
            entropy_A = -torch.sum(probs_a * torch.log(probs_a + 1e-6), dim=1)
            top2, _ = torch.topk(probs_a, k=2, dim=1)
            margin_A = top2[:, 0] - top2[:, 1]
            _, preds_c = torch.max(model_c(images), dim=1)
            actual_route_to_c = ((preds_a != labels) & (preds_c == labels)).long()
            for i in range(images.size(0)):
                rows.append({
                    'confidence': conf_A[i].item(), 'entropy': entropy_A[i].item(), 'margin': margin_A[i].item(),
                    'correct_a': (preds_a[i] == labels[i]).item(), 'correct_c': (preds_c[i] == labels[i]).item(),
                    'target_route': actual_route_to_c[i].item(),
                })
    return pd.DataFrame(rows)

def extract_3stage_telemetry(loader, model_a, model_b, model_c, device):
    rows = []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            probs_a = torch.softmax(model_a(images), dim=1)
            conf_a, preds_a = torch.max(probs_a, dim=1)
            entropy_a = -torch.sum(probs_a * torch.log(probs_a + 1e-6), dim=1)
            top2_a, _ = torch.topk(probs_a, k=2, dim=1)
            margin_a = top2_a[:, 0] - top2_a[:, 1]

            probs_b = torch.softmax(model_b(images), dim=1)
            conf_b, preds_b = torch.max(probs_b, dim=1)
            entropy_b = -torch.sum(probs_b * torch.log(probs_b + 1e-6), dim=1)
            top2_b, _ = torch.topk(probs_b, k=2, dim=1)
            margin_b = top2_b[:, 0] - top2_b[:, 1]

            _, preds_c = torch.max(model_c(images), dim=1)
            target_a_to_b = ((preds_a != labels) & (preds_b == labels)).long()
            target_b_to_c = ((preds_b != labels) & (preds_c == labels)).long()

            for i in range(images.size(0)):
                rows.append({
                    'conf_a': conf_a[i].item(), 'entropy_a': entropy_a[i].item(), 'margin_a': margin_a[i].item(),
                    'conf_b': conf_b[i].item(), 'entropy_b': entropy_b[i].item(), 'margin_b': margin_b[i].item(),
                    'correct_a': (preds_a[i] == labels[i]).item(), 'correct_b': (preds_b[i] == labels[i]).item(),
                    'correct_c': (preds_c[i] == labels[i]).item(),
                    'target_ab': target_a_to_b[i].item(), 'target_bc': target_b_to_c[i].item()
                })
    return pd.DataFrame(rows)

# --- Router training functions (identical to the main study / Jetson script) ---

def _cost_sensitive_pos_weight(y_train, latency_c, cost_missed_fallback=1.0):
    asymmetric_multiplier = cost_missed_fallback / max(latency_c, 1e-6)
    num_pos = np.sum(y_train == 1)
    num_neg = np.sum(y_train == 0)
    base_weight = num_neg / max(num_pos, 1)
    pos_weight = base_weight * asymmetric_multiplier
    return float(np.clip(pos_weight, 1.0, 10.0))

def train_xgb_router(X_train, y_train, latency_c, cost_missed_fallback=1.0, seed=42):
    pos_weight = _cost_sensitive_pos_weight(y_train, latency_c, cost_missed_fallback)
    router = xgb.XGBClassifier(n_estimators=50, max_depth=3, learning_rate=0.1,
                                scale_pos_weight=pos_weight, eval_metric='logloss', random_state=seed, n_jobs=2)
    router.fit(X_train, y_train)
    return router

def train_rf_router(X_train, y_train, latency_c, cost_missed_fallback=1.0, seed=42):
    pos_weight = _cost_sensitive_pos_weight(y_train, latency_c, cost_missed_fallback)
    router = RandomForestClassifier(n_estimators=50, max_depth=3, class_weight={0: 1.0, 1: pos_weight},
                                     random_state=seed, n_jobs=2)
    router.fit(X_train, y_train)
    return router

def train_logreg_router(X_train, y_train, latency_c, cost_missed_fallback=1.0, seed=42):
    pos_weight = _cost_sensitive_pos_weight(y_train, latency_c, cost_missed_fallback)
    router = LogisticRegression(class_weight={0: 1.0, 1: pos_weight}, max_iter=1000, random_state=seed)
    router.fit(X_train, y_train)
    return router

def train_dtree_router(X_train, y_train, latency_c, cost_missed_fallback=1.0, seed=42):
    pos_weight = _cost_sensitive_pos_weight(y_train, latency_c, cost_missed_fallback)
    router = DecisionTreeClassifier(max_depth=3, class_weight={0: 1.0, 1: pos_weight}, random_state=seed)
    router.fit(X_train, y_train)
    return router

def _weighted_accuracy_fitness(weights, X_arr, y_arr, sample_weights, theta=0.20):
    score = X_arr @ weights[:3] + weights[3]
    logit_theta = np.log(theta / (1 - theta))
    preds = (score >= logit_theta).astype(int)
    correct = (preds == y_arr).astype(float)
    return np.sum(correct * sample_weights) / np.sum(sample_weights)

def _genetic_algorithm_search(X_arr, y_arr, sample_weights, seed=42, pop_size=40, generations=60,
                               gene_range=8.0, mutation_rate=0.2, mutation_scale=0.6, theta=0.20):
    rng = np.random.default_rng(seed)
    population = rng.uniform(-gene_range, gene_range, size=(pop_size, 4))

    def fitness_of(ind):
        return _weighted_accuracy_fitness(ind, X_arr, y_arr, sample_weights, theta=theta)

    best_individual, best_fitness = None, -np.inf
    for _ in range(generations):
        fitnesses = np.array([fitness_of(ind) for ind in population])
        gen_best_idx = np.argmax(fitnesses)
        if fitnesses[gen_best_idx] > best_fitness:
            best_fitness = fitnesses[gen_best_idx]
            best_individual = population[gen_best_idx].copy()
        new_population = [best_individual.copy()]
        while len(new_population) < pop_size:
            i, j = rng.integers(0, pop_size, size=2)
            p1 = population[i] if fitnesses[i] > fitnesses[j] else population[j]
            i, j = rng.integers(0, pop_size, size=2)
            p2 = population[i] if fitnesses[i] > fitnesses[j] else population[j]
            mask = rng.random(4) < 0.5
            child = np.where(mask, p1, p2)
            mutate_mask = rng.random(4) < mutation_rate
            child = child + mutate_mask * rng.normal(0, mutation_scale, size=4)
            child = np.clip(child, -gene_range, gene_range)
            new_population.append(child)
        population = np.array(new_population[:pop_size])
    return best_individual, best_fitness

class HeuristicRouter:
    """Must stay importable under this exact name/module on the Jetson side
    too -- joblib/pickle needs the class definition available to unpickle."""
    def __init__(self, weights):
        self.weights = np.asarray(weights, dtype=float)

    def predict_proba(self, X):
        X_arr = X.values if hasattr(X, 'values') else np.asarray(X)
        score = X_arr @ self.weights[:3] + self.weights[3]
        prob_escalate = 1.0 / (1.0 + np.exp(-score))
        return np.column_stack([1.0 - prob_escalate, prob_escalate])

def train_heuristic_router(X_train, y_train, latency_c, cost_missed_fallback=1.0, seed=42, n_restarts=25):
    pos_weight = _cost_sensitive_pos_weight(y_train, latency_c, cost_missed_fallback)
    X_arr = X_train.values if hasattr(X_train, 'values') else np.asarray(X_train)
    y_arr = y_train.values if hasattr(y_train, 'values') else np.asarray(y_train)
    sample_weights = np.where(y_arr == 1, pos_weight, 1.0)
    best_weights_overall, best_fitness_overall = None, -np.inf
    for restart in range(n_restarts):
        restart_seed = seed * 1000 + restart
        weights, fitness = _genetic_algorithm_search(X_arr, y_arr, sample_weights, seed=restart_seed, theta=0.20)
        if fitness > best_fitness_overall:
            best_fitness_overall = fitness
            best_weights_overall = weights
    return HeuristicRouter(best_weights_overall)

ROUTER_TRAIN_FNS = {
    'xgb': train_xgb_router, 'rf': train_rf_router, 'logreg': train_logreg_router,
    'dtree': train_dtree_router, 'heuristic': train_heuristic_router,
}

def save_router(router, router_dir, filename):
    path = os.path.join(router_dir, filename)
    joblib.dump(router, path)
    print("  saved {}".format(path))

import time

def main():
    parser = argparse.ArgumentParser(description="Export trained routers for offline Jetson evaluation.")
    parser.add_argument('--data_dir', type=str, default='./data')
    parser.add_argument('--router_dir', type=str, default='./exported_routers')
    parser.add_argument('--n_samples', type=int, default=4000)
    parser.add_argument('--target_latency_c', type=float, default=39.44)
    parser.add_argument('--target_latency_b', type=float, default=23.27)
    args, _ = parser.parse_known_args()

    os.makedirs(args.router_dir, exist_ok=True)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print("Using device: {}".format(device))

    # --- TRACKING ROUTER FIT TIMES & TOTAL WALL-CLOCK ---
    script_start_time = time.perf_counter()
    router_train_times = {name: 0.0 for name in ROUTER_TRAIN_FNS.keys()}

    def timed_train(name, *train_args, **train_kwargs):
        t0 = time.perf_counter()
        model = ROUTER_TRAIN_FNS[name](*train_args, **train_kwargs)
        elapsed = time.perf_counter() - t0
        router_train_times[name] += elapsed
        return model

    # --- Bench routers (CIFAR-100, seed 42) ---
    print("\n=== Training bench routers (CIFAR-100, seed 42, probe data) ===")
    cifar100_transform = DATASETS['CIFAR-100']['transform']
    val_cifar100 = get_dataset('CIFAR-100', cifar100_transform, data_root=args.data_dir)
    m_a, m_b, m_c = load_models_for_dataset('CIFAR-100', device)
    subset_probe = torch.utils.data.Subset(val_cifar100, list(range(1000)))
    loader_probe = torch.utils.data.DataLoader(subset_probe, batch_size=32, shuffle=False)
    df_probe = extract_telemetry(loader_probe, m_a, m_c, device)
    X_pr, y_pr = df_probe[['confidence', 'entropy', 'margin']], df_probe['target_route']
    X_tr, X_te, y_tr, y_te = train_test_split(X_pr, y_pr, test_size=0.2, random_state=42)
    lat_c_placeholder = args.target_latency_c
    lat_b_placeholder = args.target_latency_b if args.target_latency_b is not None else args.target_latency_c

    for name in ROUTER_TRAIN_FNS.keys():
        router = timed_train(name, X_tr, y_tr, lat_c_placeholder, seed=42)
        save_router(router, args.router_dir, "bench_{}.joblib".format(name))

    # --- 2-tier routers ---
    for dname in ['CIFAR-100', 'CIFAR-10']:
        print("\n=== 2-tier routers: {} ===".format(dname))
        transform = DATASETS[dname]['transform']
        dataset = get_dataset(dname, transform, data_root=args.data_dir)
        ma, mb, mc = load_models_for_dataset(dname, device)

        for s in DEFAULT_SEEDS_2TIER:
            torch.manual_seed(s)
            idx = torch.randperm(len(dataset))[:args.n_samples]
            sub = torch.utils.data.Subset(dataset, idx)
            loader = torch.utils.data.DataLoader(sub, batch_size=32, shuffle=False)
            df = extract_telemetry(loader, ma, mc, device)
            X, y = df[['confidence', 'entropy', 'margin']], df['target_route']
            X_train, _, y_train, _ = train_test_split(X, y, test_size=0.2, random_state=s)

            for name in ROUTER_TRAIN_FNS.keys():
                router = timed_train(name, X_train, y_train, lat_c_placeholder, seed=s)
                save_router(router, args.router_dir, "2tier_{}_{}_{}.joblib".format(dname, s, name))

    # --- 3-tier routers ---
    for dname in ['CIFAR-10', 'CIFAR-100']:
        print("\n=== 3-tier routers: {} ===".format(dname))
        transform = DATASETS[dname]['transform']
        dataset = get_dataset(dname, transform, data_root=args.data_dir)
        ma, mb, mc = load_models_for_dataset(dname, device)
        loader_full = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=False)
        df_3st = extract_3stage_telemetry(loader_full, ma, mb, mc, device)

        feats_a, feats_b = ['conf_a', 'entropy_a', 'margin_a'], ['conf_b', 'entropy_b', 'margin_b']
        ren_a = {'conf_a': 'confidence', 'entropy_a': 'entropy', 'margin_a': 'margin'}
        ren_b = {'conf_b': 'confidence', 'entropy_b': 'entropy', 'margin_b': 'margin'}

        for seed in DEFAULT_SEEDS_3TIER:
            df_tr, _ = train_test_split(df_3st, test_size=0.2, random_state=seed)
            target_ac_tr = ((df_tr['correct_a'] == 0) & (df_tr['correct_c'] == 1)).astype(int)

            for name in ROUTER_TRAIN_FNS.keys():
                r_ac = timed_train(name, df_tr[feats_a].rename(columns=ren_a), target_ac_tr, lat_c_placeholder, seed=seed)
                save_router(r_ac, args.router_dir, "3tier_{}_{}_{}_ac.joblib".format(dname, seed, name))

                r_ab = timed_train(name, df_tr[feats_a].rename(columns=ren_a), df_tr['target_ab'], lat_b_placeholder, seed=seed)
                save_router(r_ab, args.router_dir, "3tier_{}_{}_{}_ab.joblib".format(dname, seed, name))

                r_bc = timed_train(name, df_tr[feats_b].rename(columns=ren_b), df_tr['target_bc'], lat_c_placeholder, seed=seed)
                save_router(r_bc, args.router_dir, "3tier_{}_{}_{}_bc.joblib".format(dname, seed, name))

    # --- PRINT CUMULATIVE ROUTER COMPUTE TIME TABLE ---
    total_wall_clock = time.perf_counter() - script_start_time
    total_router_fit_time = sum(router_train_times.values())

    label_map = {
        'xgb': 'XGBoost',
        'rf': 'Random Forest',
        'logreg': 'Logistic Regression',
        'dtree': 'Decision Tree',
        'heuristic': 'GA Heuristic'
    }

    print("\n" + "=" * 50)
    print("TABLE VI: CUMULATIVE ROUTER COMPUTE TIME")
    print("=" * 50)
    print(f"{'Router Family':<22} | {'Time (s)':<10} | {'Share (%)':<10}")
    print("-" * 50)
    for key, pretty_name in label_map.items():
        t = router_train_times[key]
        share = (t / total_router_fit_time * 100) if total_router_fit_time > 0 else 0
        print(f"{pretty_name:<22} | {t:<10.2f} | {share:<10.1f}")
    print("-" * 50)
    print(f"Total wall-clock: {total_wall_clock:.2f} s ({total_wall_clock/60:.2f} min)")
    print("=" * 50 + "\n")

    # Also save to CSV for paper reproducibility
    df_compute = pd.DataFrame([
        {'Router Family': label_map[k], 'Time (s)': round(router_train_times[k], 2),
         'Share (%)': round((router_train_times[k]/total_router_fit_time*100), 1)}
        for k in label_map.keys()
    ])
    df_compute.to_csv(os.path.join(args.router_dir, 'cumulative_compute_time.csv'), index=False)

if __name__ == '__main__':
    main()

Using device: cuda

=== Training bench routers (CIFAR-100, seed 42, probe data) ===


100%|██████████| 169M/169M [00:59<00:00, 2.84MB/s]


The repository chenyaofo_pytorch-cifar-models does not belong to the list of trusted repositories and as such cannot be downloaded. Do you trust this repository and wish to add it to the trusted list of repositories (y/N)?y
Downloading: "https://github.com/chenyaofo/pytorch-cifar-models/zipball/master" to /root/.cache/torch/hub/master.zip
Downloading: "https://github.com/chenyaofo/pytorch-cifar-models/releases/download/resnet/cifar100_resnet20-23dac2f1.pt" to /root/.cache/torch/hub/checkpoints/cifar100_resnet20-23dac2f1.pt


100%|██████████| 1.11M/1.11M [00:00<00:00, 127MB/s]
Using cache found in /root/.cache/torch/hub/chenyaofo_pytorch-cifar-models_master


Downloading: "https://github.com/chenyaofo/pytorch-cifar-models/releases/download/resnet/cifar100_resnet32-84213ce6.pt" to /root/.cache/torch/hub/checkpoints/cifar100_resnet32-84213ce6.pt


...

  saved ./exported_routers/2tier_CIFAR-10_1_xgb.joblib
  saved ./exported_routers/2tier_CIFAR-10_1_rf.joblib
  saved ./exported_routers/2tier_CIFAR-10_1_logreg.joblib
  saved ./exported_routers/2tier_CIFAR-10_1_dtree.joblib
  saved ./exported_routers/2tier_CIFAR-10_1_heuristic.joblib
  saved ./exported_routers/2tier_CIFAR-10_2_xgb.joblib
  saved ./exported_routers/2tier_CIFAR-10_2_rf.joblib
  saved ./exported_routers/2tier_CIFAR-10_2_logreg.joblib
  saved ./exported_routers/2tier_CIFAR-10_2_dtree.joblib
  saved ./exported_routers/2tier_CIFAR-10_2_heuristic.joblib
  saved ./exported_routers/2tier_CIFAR-10_3_xgb.joblib
  saved ./exported_routers/2tier_CIFAR-10_3_rf.joblib
  saved ./exported_routers/2tier_CIFAR-10_3_logreg.joblib
  saved ./exported_routers/2tier_CIFAR-10_3_dtree.joblib
  saved ./exported_routers/2tier_CIFAR-10_3_heuristic.joblib
  saved ./exported_routers/2tier_CIFAR-10_4_xgb.joblib
  saved ./exported_routers/2tier_CIFAR-10_4_rf.joblib
  saved ./exported_routers/2tier_CIF

Using cache found in /root/.cache/torch/hub/chenyaofo_pytorch-cifar-models_master
Using cache found in /root/.cache/torch/hub/chenyaofo_pytorch-cifar-models_master
Using cache found in /root/.cache/torch/hub/chenyaofo_pytorch-cifar-models_master


  saved ./exported_routers/3tier_CIFAR-10_42_xgb_ac.joblib
  saved ./exported_routers/3tier_CIFAR-10_42_xgb_ab.joblib
  saved ./exported_routers/3tier_CIFAR-10_42_xgb_bc.joblib
  saved ./exported_routers/3tier_CIFAR-10_42_rf_ac.joblib
  saved ./exported_routers/3tier_CIFAR-10_42_rf_ab.joblib
  saved ./exported_routers/3tier_CIFAR-10_42_rf_bc.joblib
  saved ./exported_routers/3tier_CIFAR-10_42_logreg_ac.joblib
  saved ./exported_routers/3tier_CIFAR-10_42_logreg_ab.joblib
  saved ./exported_routers/3tier_CIFAR-10_42_logreg_bc.joblib
  saved ./exported_routers/3tier_CIFAR-10_42_dtree_ac.joblib
  saved ./exported_routers/3tier_CIFAR-10_42_dtree_ab.joblib
  saved ./exported_routers/3tier_CIFAR-10_42_dtree_bc.joblib
  saved ./exported_routers/3tier_CIFAR-10_42_heuristic_ac.joblib
  saved ./exported_routers/3tier_CIFAR-10_42_heuristic_ab.joblib
  saved ./exported_routers/3tier_CIFAR-10_42_heuristic_bc.joblib
  saved ./exported_routers/3tier_CIFAR-10_43_xgb_ac.joblib
  saved ./exported_routers

Using cache found in /root/.cache/torch/hub/chenyaofo_pytorch-cifar-models_master
Using cache found in /root/.cache/torch/hub/chenyaofo_pytorch-cifar-models_master
Using cache found in /root/.cache/torch/hub/chenyaofo_pytorch-cifar-models_master


  saved ./exported_routers/3tier_CIFAR-100_42_xgb_ac.joblib
  saved ./exported_routers/3tier_CIFAR-100_42_xgb_ab.joblib
  saved ./exported_routers/3tier_CIFAR-100_42_xgb_bc.joblib
  saved ./exported_routers/3tier_CIFAR-100_42_rf_ac.joblib
  saved ./exported_routers/3tier_CIFAR-100_42_rf_ab.joblib
  saved ./exported_routers/3tier_CIFAR-100_42_rf_bc.joblib
  saved ./exported_routers/3tier_CIFAR-100_42_logreg_ac.joblib
  saved ./exported_routers/3tier_CIFAR-100_42_logreg_ab.joblib
  saved ./exported_routers/3tier_CIFAR-100_42_logreg_bc.joblib
  saved ./exported_routers/3tier_CIFAR-100_42_dtree_ac.joblib
  saved ./exported_routers/3tier_CIFAR-100_42_dtree_ab.joblib
  saved ./exported_routers/3tier_CIFAR-100_42_dtree_bc.joblib
  saved ./exported_routers/3tier_CIFAR-100_42_heuristic_ac.joblib
  saved ./exported_routers/3tier_CIFAR-100_42_heuristic_ab.joblib
  saved ./exported_routers/3tier_CIFAR-100_42_heuristic_bc.joblib
  saved ./exported_routers/3tier_CIFAR-100_43_xgb_ac.joblib
  saved ./

In [ ]:
import shutil
from google.colab import files

# Zip the exported_routers directory
shutil.make_archive('exported_routers', 'zip', '/content/exported_routers')

# Download the zip file
files.download('exported_routers.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import sklearn, xgboost
print(sklearn.__version__, xgboost.__version__)

1.9.0 3.2.0
